<h2>读入数据集并分析</h2>

In [ ]:
import numpy as np
import pandas as pd

data2=pd.read_csv(r"wdbc.data",',',encoding='utf-8')   #读取数据集
#显示数据集的前五条
print(data2.head())
print('删除前的数据集大小为:{}'.format(data2.shape))
data2.drop_duplicates(inplace=True)  #删除重复数据
# data.replace('?', pd.NA, inplace=True)
# data.dropna(inplace=True)   #删除缺失值
print('删除后的数据集大小为:{}'.format(data2.shape))
header=range(32)
data2.columns=header
data2=data2.drop(columns=[0])  #删除第1列
header=range(31)
data2.columns=header
# data2.to_excel('wdbc.xlsx', sheet_name='data') #写到excel里方便分析一下

In [ ]:
data2.replace({'M': 1, 'B': 0}, inplace=True)
data2[0].value_counts()

<h2>评估指标相关函数</h2>

<h3>一、定义绘制混淆矩阵的函数</h3>

In [ ]:
from sklearn import metrics
from sklearn.metrics import classification_report
import itertools
import matplotlib.pyplot as plt
import numpy as np
# 绘制混淆矩阵
def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)
    plt.ylim(len(classes) - 0.5, -0.5)
    fmt =  'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

<h3>二、定义绘制ROC曲线的函数</h3>

In [ ]:
from sklearn.metrics import roc_curve, auc
def plot_roc_curve(y,score):
    fpr, tpr, thresholds = roc_curve(y, score)

    # 计算AUC
    roc_auc = auc(fpr, tpr)

    # 绘制ROC曲线
    plt.title('Receiver Operating Characteristic')
    plt.plot(fpr, tpr, 'b', label = 'AUC = %0.4f' % roc_auc)
    plt.legend(loc = 'lower right')
    plt.plot([0, 1], [0, 1],'r--')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.show()

<h2>事前准备：创建训练集和测试集</h2>

In [ ]:
n=0.8
data2 = data2.apply(pd.to_numeric)
classnumber=data2.value_counts()

bad=data2[data2[0]==1]
good=data2[data2[0]==0]

bad=bad.sample(len(bad),random_state=0)
good=good.sample(len(good),random_state=0)

#分层抽样后相连,创建训练集和测试集
train_X=pd.concat([bad.iloc[:int(len(bad)*0.8),1:],good.iloc[:int(len(good)*0.8),1:]],axis=0)
train_y=pd.concat([bad.iloc[:int(len(bad)*0.8),0],good.iloc[:int(len(good)*0.8),0]],axis=0)
test_X=pd.concat([bad.iloc[int(len(bad)*0.8):,1:],good.iloc[int(len(good)*0.8):,1:]],axis=0)
test_y=pd.concat([bad.iloc[int(len(bad)*0.8):,0],good.iloc[int(len(good)*0.8):,0]],axis=0)

#做归一化处理后再进行knn训练
# mins = np.min(train_X, axis=0)
# maxs = np.max(train_X, axis=0)
# train_X = (train_X - mins) / (maxs - mins)
#
# mins = np.min(test_X, axis=0)
# maxs = np.max(test_X, axis=0)
# test_X = (test_X - mins) / (maxs - mins)

In [ ]:
X_train=np.asarray(train_X)
y_train=np.asarray(train_y)
X_test=np.asarray(test_X)
y_test=np.asarray(test_y)

In [ ]:
accuracy=[] #存放每个特征的准确率
k=7

for i in range(X_train.shape[1]):
    result=[]   #存放预测结果
    for x in X_test:
        dis=(x-X_train) ** 2
        dis=dis.transpose()
        subdis=dis[i]
        index=subdis.argsort()     #返回数组排序后数组元素的索引
        index=index[:k]
        count=np.bincount(y_train[index])      #返回数组中元素出现的次数，必须是自然数
        result.append(count.argmax())     #返回出现最多元素的索引
    num_correct = np.sum(result == y_test)
    accuracy .append( num_correct / len(y_test))

index = [i for i, x in enumerate(accuracy) if x > 0.8]
print(accuracy)
print(index)

In [ ]:
subXtrain=X_train[:,index]
subXtest=X_test[:,index]

<h1>一、KNN算法的实现</h1>可以多分类，数值特征即可

<h3>knn类的编写</h3>

In [ ]:
class KNN:
    #初始化方法，k为选择的邻居个数
    def __init__(self,k):
        self.k=k

    def fit(self,X,y):
        """
        训练方法，在KNN算法中仅仅是转换数据类型格式，使得传进来的X和y可以是多种数据类型
        :parameters
        -----
        X:训练样本，形状为[样本数，特征数]
        y:训练样本的实际值，形状为[样本数]
        """
        self.X=np.asarray(X)
        self.y=np.asarray(y)

    def predict(self,X):
        """
        预测方法
        :param X:预测样本，形状为[样本数，特征数]
        -----
        :return:
        -----
         result:预测结果，是测试集预测结果的合集
        """
        X=np.asarray(X)
        result=[]
        score=[]
        for x in X:
            dis=np.sum((x-self.X) ** 2,axis=1)     #计算距离，此时每行元素即为测试样本到训练样本的距离的平方
            index=dis.argsort()     #返回数组排序后数组元素的索引
            index=index[:self.k]
            count=np.bincount(self.y[index])      #返回数组中元素出现的次数，必须是自然数
            result.append(count.argmax())     #返回出现最多元素的索引
            score.append(count[0]/self.k)

        return np.asarray(result),score

<h3>knn类的训练</h3>

In [ ]:
knn=KNN(k=7)
knn.fit(train_X,train_y)
result,score=knn.predict(test_X)

In [ ]:
knn=KNN(k=7)
knn.fit(subXtrain,train_y)
result,score=knn.predict(subXtest)

<h3>输出评价指标</h3>

In [ ]:
from sklearn import metrics
from sklearn.metrics import classification_report

matrix=metrics.confusion_matrix(test_y, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(test_y, result))

In [ ]:
y_test=np.asarray(test_y)
score2=[1-x for x in score]
plot_roc_curve(y_test,score2)

<font color="red">可以看到准确率很高，说明10个特征基本都是有用的，且权值大致相等，分布集中，所以分类效果较好

<h1>二、朴素贝叶斯决策算法的实现</h1>多分类应该也可以，该贝叶斯是多项式贝叶斯，输入需为离散量，要解决连续问题或许另写高斯贝叶斯

<h3>朴素贝叶斯类的编写</h3>

In [ ]:
class NaiveBayes:

    def __init__(self):

        self.lablepl={}  #用来存放类条件概率的字典{类别名：出现概率}
        self.features=[]    #计算每个特征出现的概率

    def fit(self,X_train,y_train):
        self.X_train=np.asarray(X_train)
        self.y_train=np.asarray(y_train)

        #1、计算类条件概率
        # 统计元素出现的次数
        self.labels, self.counts = np.unique(self.y_train, return_counts=True)
        # 将元素名和元素个数转换为字典格式
        count_dict = dict(zip(self.labels, self.counts))
        for k,v in count_dict.items():
            self.lablepl[k]=v/len(self.y_train)

        #2、计算每个特征出现的概率
        numberf=len(self.X_train[0])
        for label in self.labels:
            feature=[]
            for j in range(numberf):    #按照特征数往feature里加入空字典
                feature.append({})
            for i in range(len(self.y_train)):   #取出一条数据
                if self.y_train[i]==label:
                    for k in range(len(self.X_train[0])):
                        if self.X_train[i][k] not in feature[k]:
                            feature[k][self.X_train[i][k]]=1
                        else:
                            feature[k][self.X_train[i][k]]=feature[k][self.X_train[i][k]]+1
            self.features.append(feature)

        for i in range(len(self.labels)):
            for j in range(len(self.X_train[0])):
                for key in self.features[i][j]:
                    self.features[i][j][key]=self.features[i][j][key]/self.counts[i]

    def predict(self,X_test):
        self.X_test=np.asarray(X_test)
        result=[]
        possiblity=[]
        # for i in range(len(y_test)):    #取出一条测试样本
        for x in self.X_test:
            possible=[]
            for j in range(len(self.labels)):    #分别计算两种标签的概率
                sum=0
                for k in range(len(self.X_test[0])):
                    if x[k] in self.features[j][k]:  #如果测试集的值有在训练集出现过就取出对应概率
                        sum=sum+np.log(self.lablepl[self.labels[j]])  #labels的下标对应的值是labelpl的key
                    else:   #否则按平滑处理
                        sum=sum+np.log(1.0/self.counts[j])
                sum=sum+np.log(self.lablepl[self.labels[j]])
                possible.append(sum)
            possiblity.append(possible)
            index=np.argmax(possible)
            result.append(self.labels[index])
        return result,possiblity

<h3>训练贝叶斯分类器</h3>

In [ ]:
nbclasser=NaiveBayes()
nbclasser.fit(train_X,train_y)
result,possiblity=nbclasser.predict(test_X)

In [ ]:
matrix=metrics.confusion_matrix(test_y, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(test_y, result))

In [ ]:
score=[]
for i in range(len(y_test)):
    score.append(possiblity[i][0]/(possiblity[i][0]+possiblity[i][1]))
plot_roc_curve(y_test,score)

可以看到准确率并不高，2能够百分百准确，但4不行，考虑应该是因为该朴素贝叶斯是离散式的朴素贝叶斯，有些特征值在训练集出现了，但在测试集并未出现，所以此时类概率就成了判定因素，但2类又远远高于4类，导致产生了这种情况。

<h1>二（改）、以高斯分布估计概率的朴素贝叶斯实现<h1>

In [ ]:
#1、定义sigmoid函数
def sigmoid(x):
    return  1 / (1 + np.exp(-x))

In [ ]:
X_train=np.asarray(train_X)
y_train=np.asarray(train_y)
X_test=np.asarray(test_X)
y_test=np.asarray(test_y)

# X_train=10*X_train
# X_test=10*X_test

In [ ]:
import math
def normal_distribution(x,mean,variance):
    a1=math.exp(-((x - mean) ** 2) / (2 * variance))
    a2=math.sqrt(2 * variance * math.pi)
    if a2==0:
        return 0.5
    else:
        if (1/a2)*a1 <1e-10:
            return 1e-10
        else:
            return (1/a2)*a1

class GNaiveBayes:

    def __init__(self):

        self.lablepl={}  #用来存放类条件概率的字典{类别名：出现概率}
        self.dic={}  #键为标签，值为一个存有均值和方差的列表


    def fit(self,X_train,y_train):
        self.X_train=np.asarray(X_train)
        self.y_train=np.asarray(y_train)

        #1、计算类条件概率
        # 统计元素出现的次数
        self.labels, self.counts = np.unique(self.y_train, return_counts=True)
        # 将元素名和元素个数转换为字典格式
        count_dict = dict(zip(self.labels, self.counts))
        for k,v in count_dict.items():
            self.lablepl[k]=v/len(self.y_train)

        #2、每列特征出现的均值和方差
        numberf=len(self.X_train[0])
        for label in self.labels:
            subx=X_train[y_train==label]
            he=[]   #第一个元素为均值，第二个元素为方差
            means=np.mean(subx,axis=0)
            he.append(means)
            variance = np.var(subx, axis=0,ddof=1)
            he.append(variance)
            self.dic[label]=he

    def predict(self,X_test):
        self.X_test=np.asarray(X_test)
        result=[]
        score=[]
        for x in self.X_test:   #取出一条测试样本
            possible=[]
            # print(x)
            for key,value in self.dic.items():
                sum=0
                for i in range(len(x)):
                    sum+=np.log(normal_distribution(x[i],value[0][i],value[1][i]))
                sum=sum+np.log(self.lablepl[key])
                possible.append(sum)
            index=np.argmax(possible)
            result.append(index)
            score.append(sigmoid(possible[1]-(possible[0]+possible[1])/2))

        return result,score

nbclasser=GNaiveBayes()
nbclasser.fit(X_train,y_train)
result,score=nbclasser.predict(X_test)

In [ ]:
matrix=metrics.confusion_matrix(test_y, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(test_y, result))

In [ ]:
print(result)

In [ ]:
plot_roc_curve(y_test,score)

<h1>五、逻辑回归的实现</h1>

In [ ]:
X_train=np.asarray(train_X)
y_train=np.asarray(train_y)
X_test=np.asarray(test_X)
y_test=np.asarray(test_y)

# #重新赋值，把恶性赋值为1，把良性赋值为0
# y_train=np.where(y_train==4,1,y_train)
# y_train=np.where(y_train==2,0,y_train)
# y_test=np.where(y_test==4,1,y_test)
# y_test=np.where(y_test==2,0,y_test)
# print(y_test)

#X都各加一列作为偏置
X_train = np.c_[np.ones(X_train.shape[0]),X_train]
X_test = np.c_[np.ones(X_test.shape[0]),X_test]
print(X_test.shape)

# subXtrain = np.c_[np.ones(subXtrain.shape[0]),subXtrain]
# subXtest = np.c_[np.ones(subXtest.shape[0]),subXtest]
# print(subXtest)

In [ ]:
#1、定义sigmoid函数
def sigmoid(x):
    return  1 / (1 + np.exp(-x))

In [ ]:
#定义训练函数
learning_rate=0.1  #设置学习率
num_iterations=1000   #设置迭代次数
threshold=0.00000003     #设置阈值，当梯度小于阈值的时候就会停止训练

def lrfit(X, y, learning_rate=0.1, num_iterations=1000, threshold=0.0003):

        #初始化权重
        theta = np.ones(X.shape[1])

        #初始化梯度值
        gradient=np.zeros(X.shape[1])

        #开始梯度下降训练
        for n in range(num_iterations):
            for j in range(X.shape[1]): #遍历每个特征
                jf=0

                for i in range(X.shape[0]): #遍历每个样本
                    z=np.dot(X[i],theta)
                    h=sigmoid(z)
                    jf+=(h-y[i])*X[i][j]

                jf/=X.shape[0]
                gradient[j]=jf

            theta=theta-learning_rate*gradient
            if abs(gradient.sum()) <threshold:
                return theta

        return theta

In [ ]:
def lrpredict(X_test,theta):
    result=[]
    score=[]
    for x in X_test:
        z=sigmoid(np.dot(x,theta))
        if z>0.5:
            result.append(1)
        else:
            result.append(0)
        score.append(z)
    return result,score

In [ ]:
theta=lrfit(X_train,y_train,num_iterations=6000)
print(theta)

In [ ]:
result,score=lrpredict(X_test,theta)

In [ ]:
matrix=metrics.confusion_matrix(y_test, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(y_test, result))

In [ ]:
plot_roc_curve(y_test,score)

<h1>特征选择</h1>

In [ ]:
featureX=data2.drop(columns=[0])
featurey=data2[0]
featureX=np.asarray(featureX)
featurey=np.asarray(featurey)

mins = np.min(featureX, axis=0)
maxs = np.max(featureX, axis=0)
featureX_scaled = (featureX - mins) / (maxs - mins)

In [ ]:
def relief(X, y):
    n_samples = X.shape[0]
    n_features =X.shape[1]
    weights = np.zeros(n_features)

    for i in range(n_samples):
        # 找到样本最近的两类邻居
        dis=np.sum((X[i]-X) ** 2,axis=1)     #计算距离，此时每行元素即为测试样本到训练样本的距离的平方
        nearest_neighbor=dis.argsort()     #返回数组排序后数组元素的索引
        same=0
        diffrent=0
        for k in range(len(nearest_neighbor)):
            if y[i]==y[nearest_neighbor[k]] and same<=1:
                if same==0: #排除自己与自己距离最近的情况
                    same=1
                    continue
                for j in range(n_features):
                    weights[j] -= abs(X[i, j] - X[nearest_neighbor[k], j])
                same=2
            elif y[i]!=y[nearest_neighbor[k]] and diffrent ==0:
                 for j in range(n_features):
                    weights[j] += abs(X[i, j] - X[nearest_neighbor[k], j])
                    diffrent=1
            if same==2 and diffrent==1:
                break
        # 更新i的特征的权重
        # for j in range(n_features):
        #     if y[i] == y[nearest_neighbor]:
        #         weights[j] += (X[i, j] - X[nearest_neighbor, j])**2
        #     else:
        #         weights[j] -= (X[i, j] - X[nearest_neighbor, j])**2

    top_k = np.argsort(weights)
    # top_k = weights

    return top_k

top=relief(featureX_scaled,featurey)

In [ ]:
print(top)

In [ ]:
top15=top[-15:]
print(top15)
subXtrain=X_train[:,top15]
subXtest=X_test[:,top15]

In [ ]:
knn=KNN(k=7)
knn.fit(subXtrain,train_y)
result,score=knn.predict(subXtest)

In [ ]:
print(result)

In [ ]:
matrix=metrics.confusion_matrix(test_y, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(test_y, result))

In [ ]:
nbclasser=GNaiveBayes()
nbclasser.fit(subXtrain,y_train)
result,score=nbclasser.predict(subXtest)

In [ ]:
matrix=metrics.confusion_matrix(test_y, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(test_y, result))

In [ ]:
subXtrain = np.c_[np.ones(X_train.shape[0]),subXtrain]
subXtest = np.c_[np.ones(X_test.shape[0]),subXtest]

In [ ]:
theta=lrfit(subXtrain,y_train,num_iterations=6000)
print(theta)

In [ ]:
result,score=lrpredict(subXtest,theta)

In [ ]:
matrix=metrics.confusion_matrix(test_y, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(test_y, result))

In [ ]:
#集成学习，集成KNN、贝叶斯决策和KNN
def three(X_train,y_train,X_test):
    knn=KNN(k=7)
    knn.fit(X_train,y_train)
    result1,score1=knn.predict(X_test)
    nbclasser=GNaiveBayes()
    nbclasser.fit(X_train,y_train)
    result2,score2=nbclasser.predict(X_test)
    theta=lrfit(X_train,y_train,num_iterations=6000)
    result3,score3=lrpredict(X_test,theta)
    result=[]
    for i in range(len(result1)):
        tmp=[result1[i],result2[i],result3[i]]
        unique, counts = np.unique(tmp, return_counts=True)
        index=counts.argmax()
        result.append(unique[index])

    return result

In [ ]:
result=three(X_train,y_train,X_test)

In [ ]:
matrix=metrics.confusion_matrix(test_y, result, labels=None, sample_weight=None)
classes=['0','1']
plot_confusion_matrix(matrix, classes, title='Confusion matrix', cmap=plt.cm.Blues)
# y_true是真实的类别标签，y_pred是预测的类别标签
print(classification_report(test_y, result))